# Ejercicio 3. Robot con tres sensores de distancia

Se simula un robot móvil en un espacio bidimensional con cuatro obstáculos distribuidos aleatoriamente. El robot posee sensores izquierdo, central y derecho, cada uno con tres estados de distancia.

## Regla de control

Estado 0: obstáculo entre 0 y 8 unidades. Estado 1: entre 8 y 16 unidades. Estado 2: más de 16 unidades. El robot avanza cuando el frente está libre y gira cuando detecta obstáculos.

## Resultado

La trayectoria se grafica y se verifica que el robot no colisione con los obstáculos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(20)

L = 100
pasos = 500
numero_obstaculos = 4
radio_robot = 2
radio_obstaculo = 4
distancia_sensor = 15

# Direcciones: 0=arriba, 1=derecha, 2=abajo, 3=izquierda
direccion = 1
posicion = np.array([10.0, 50.0])

# Obstáculos aleatorios
obstaculos = []
while len(obstaculos) < numero_obstaculos:
    p = np.random.uniform(10, L - 10, 2)
    if np.linalg.norm(p - posicion) > 15:
        obstaculos.append(p)

def vector_direccion(d):
    return [
        np.array([0, 1]),
        np.array([1, 0]),
        np.array([0, -1]),
        np.array([-1, 0])
    ][d]

def distancia_a_obstaculos(punto):
    return min(np.linalg.norm(punto - obs) for obs in obstaculos)

def estado_sensor(distancia):
    if distancia <= 8:
        return 0
    elif distancia <= 16:
        return 1
    return 2

def leer_sensores(posicion, direccion):
    frente = vector_direccion(direccion)
    izquierda = vector_direccion((direccion - 1) % 4)
    derecha = vector_direccion((direccion + 1) % 4)

    posiciones = [
        posicion + izquierda * distancia_sensor,
        posicion + frente * distancia_sensor,
        posicion + derecha * distancia_sensor
    ]
    return [estado_sensor(distancia_a_obstaculos(p)) for p in posiciones]

def regla_robot(izq, centro, der):
    if izq == 2 and centro == 2 and der == 2:
        return "avanzar"
    if izq <= 1 and der == 2:
        return "derecha"
    if der <= 1 and izq == 2:
        return "izquierda"
    if centro <= 1:
        return "izquierda" if izq > der else "derecha"
    return "avanzar"


In [ ]:
trayectoria = [posicion.copy()]
acciones = []

for _ in range(pasos):
    sensores = leer_sensores(posicion, direccion)
    accion = regla_robot(*sensores)
    acciones.append(accion)

    if accion == "avanzar":
        posicion += vector_direccion(direccion)
    elif accion == "derecha":
        direccion = (direccion + 1) % 4
    elif accion == "izquierda":
        direccion = (direccion - 1) % 4

    posicion[0] = np.clip(posicion[0], 1, L - 1)
    posicion[1] = np.clip(posicion[1], 1, L - 1)
    trayectoria.append(posicion.copy())

trayectoria = np.array(trayectoria)

# Verificación de colisiones
distancia_minima = min(
    np.min(np.linalg.norm(trayectoria - obs, axis=1))
    for obs in obstaculos
)
colision = distancia_minima <= (radio_robot + radio_obstaculo)

print("¿Hubo colisión?:", colision)
print("Distancia mínima a un obstáculo:", distancia_minima)


In [ ]:
plt.figure(figsize=(8, 8))

for i, obs in enumerate(obstaculos):
    plt.scatter(obs[0], obs[1], s=500, marker="s",
                label="Obstáculo" if i == 0 else None)

plt.plot(trayectoria[:, 0], trayectoria[:, 1],
         label="Trayectoria del robot")

plt.scatter(trayectoria[0, 0], trayectoria[0, 1],
            s=100, marker="o", label="Inicio")

plt.scatter(trayectoria[-1, 0], trayectoria[-1, 1],
            s=100, marker="x", label="Final")

plt.xlim(0, L)
plt.ylim(0, L)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Trayectoria del robot con tres sensores")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Mostrar las últimas lecturas de sensores
print("Últimos estados de los sensores [izquierda, centro, derecha]:")
for s in [leer_sensores(trayectoria[-1], direccion)]:
    print(s)
